# Task 1: Exploratory Data Analysis (EDA)

**Nova Financial News Sentiment Analysis | KAIM Week 1**

## Objective
Perform exploratory data analysis on the financial news dataset to understand:
- Data structure and quality
- News headline patterns and trends
- Stock coverage distribution
- Temporal patterns in news publication
- Sentiment distribution (if available)

## 1. Import Libraries and Setup

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries imported successfully!")

## 2. Load the Data

In [ ]:
# Load the raw analyst ratings dataset
try:
    df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print("Error: raw_analyst_ratings.csv not found in data/raw/ folder")
    print("Please ensure the data file is in the correct location")

## 3. Initial Data Inspection

In [ ]:
# Display first few rows
print("First 5 rows:")
display(df.head())

# Display data information
print("\nData Info:")
df.info()

In [ ]:
# Basic statistics
print("Basic Statistics:")
display(df.describe(include='all'))

# Check for missing values
print("\nMissing Values:")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
display(missing_df)

## 4. Data Cleaning and Preprocessing

In [ ]:
# Convert date columns to datetime if they exist
date_columns = ['date', 'created_at', 'published_at', 'timestamp']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"Converted {col} to datetime")

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows")
    print(f"New shape: {df.shape}")

## 5. Exploratory Analysis - Headlines

In [ ]:
# Analyze headline length
if 'headline' in df.columns:
    df['headline_length'] = df['headline'].str.len()
    
    plt.figure(figsize=(12, 6))
    plt.hist(df['headline_length'], bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Headline Length')
    plt.xlabel('Number of Characters')
    plt.ylabel('Frequency')
    plt.axvline(df['headline_length'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["headline_length"].mean():.1f}')
    plt.legend()
    plt.show()
    
    print(f"Headline Length Statistics:")
    print(df['headline_length'].describe())

In [ ]:
# Most common words in headlines
if 'headline' in df.columns:
    from collections import Counter
    import re
    
    # Clean and tokenize headlines
    all_headlines = ' '.join(df['headline'].dropna().astype(str))
    words = re.findall(r'\b\w+\b', all_headlines.lower())
    
    # Remove common stop words
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 
                  'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'will', 
                  'would', 'could', 'should', 'may', 'might', 'can', 'has', 'have', 'had', 
                  'this', 'that', 'these', 'those', 'it', 'its', 'they', 'their', 'them', 
                  'he', 'she', 'his', 'her', 'him', 'we', 'us', 'our', 'you', 'your'}
    
    filtered_words = [word for word in words if word not in stop_words and len(word) > 2]
    word_counts = Counter(filtered_words)
    
    # Plot top 20 words
    top_words = word_counts.most_common(20)
    
    plt.figure(figsize=(14, 8))
    words, counts = zip(*top_words)
    plt.barh(words, counts)
    plt.title('Top 20 Most Common Words in Headlines')
    plt.xlabel('Frequency')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Top 20 most common words:")
    for word, count in top_words:
        print(f"{word}: {count}")

## 6. Stock Coverage Analysis

In [ ]:
# Analyze stock coverage if stock column exists
if 'stock' in df.columns:
    stock_counts = df['stock'].value_counts()
    
    plt.figure(figsize=(14, 8))
    stock_counts.head(20).plot(kind='bar')
    plt.title('Top 20 Most Mentioned Stocks')
    plt.xlabel('Stock Symbol')
    plt.ylabel('Number of Mentions')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f"Total unique stocks: {df['stock'].nunique()}")
    print(f"\nTop 20 stocks by mention count:")
    print(stock_counts.head(20))
else:
    print("No stock column found in the dataset")

## 7. Temporal Analysis

In [ ]:
# Time-based analysis if date column exists
date_col = None
for col in ['date', 'created_at', 'published_at', 'timestamp']:
    if col in df.columns:
        date_col = col
        break

if date_col:
    # Extract time components
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['day_of_week'] = df[date_col].dt.day_name()
    
    # News volume over time
    plt.figure(figsize=(16, 10))
    
    plt.subplot(2, 2, 1)
    yearly_counts = df['year'].value_counts().sort_index()
    yearly_counts.plot(kind='bar')
    plt.title('News Volume by Year')
    plt.xlabel('Year')
    plt.ylabel('Number of Articles')
    plt.xticks(rotation=45)
    
    plt.subplot(2, 2, 2)
    monthly_counts = df['month'].value_counts().sort_index()
    monthly_counts.plot(kind='bar')
    plt.title('News Volume by Month')
    plt.xlabel('Month')
    plt.ylabel('Number of Articles')
    
    plt.subplot(2, 2, 3)
    dow_counts = df['day_of_week'].value_counts()
    dow_counts.plot(kind='bar')
    plt.title('News Volume by Day of Week')
    plt.xlabel('Day of Week')
    plt.ylabel('Number of Articles')
    plt.xticks(rotation=45)
    
    plt.subplot(2, 2, 4)
    # Daily volume over time (sample for performance)
    daily_counts = df[date_col].dt.date.value_counts().sort_index()
    if len(daily_counts) > 1000:
        # Sample every nth point for visualization
        sample_step = len(daily_counts) // 1000
        daily_counts_sampled = daily_counts.iloc[::sample_step]
        daily_counts_sampled.plot(figsize=(8, 4))
    else:
        daily_counts.plot(figsize=(8, 4))
    plt.title('Daily News Volume Over Time')
    plt.xlabel('Date')
    plt.ylabel('Number of Articles')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Date range: {df[date_col].min()} to {df[date_col].max()}")
    print(f"Total days covered: {(df[date_col].max() - df[date_col].min()).days}")
else:
    print("No date column found in the dataset")

## 8. Sentiment Analysis (if sentiment data available)

In [ ]:
# Check for sentiment-related columns
sentiment_columns = [col for col in df.columns if 'sentiment' in col.lower()]
print(f"Sentiment-related columns found: {sentiment_columns}")

if sentiment_columns:
    for col in sentiment_columns:
        plt.figure(figsize=(12, 6))
        
        if df[col].dtype in ['int64', 'float64']:
            # Numeric sentiment
            plt.hist(df[col].dropna(), bins=50, alpha=0.7, edgecolor='black')
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Frequency')
        else:
            # Categorical sentiment
            sentiment_counts = df[col].value_counts()
            sentiment_counts.plot(kind='bar')
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{col} statistics:")
        print(df[col].describe())
else:
    print("No sentiment columns found. You may need to compute sentiment scores in Task 2.")

## 9. Data Quality Assessment

In [ ]:
# Comprehensive data quality report
print("DATA QUALITY REPORT")
print("=" * 50)

print(f"Dataset Shape: {df.shape}")
print(f"Total Records: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print()

# Column-wise analysis
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    unique_count = df[col].nunique()
    
    print(f"Column: {col}")
    print(f"  - Data Type: {df[col].dtype}")
    print(f"  - Missing Values: {missing_count:,} ({missing_pct:.2f}%)")
    print(f"  - Unique Values: {unique_count:,}")
    
    if df[col].dtype == 'object':
        print(f"  - Sample Values: {list(df[col].dropna().unique()[:5])}")
    print()

## 10. Key Findings and Summary

In [ ]:
# Generate summary statistics
print("TASK 1 EDA SUMMARY")
print("=" * 50)

print(f"1. Dataset Overview:")
print(f"   - Total records: {df.shape[0]:,}")
print(f"   - Total columns: {df.shape[1]}")
print(f"   - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()

if 'headline' in df.columns:
    print(f"2. Headline Analysis:")
    print(f"   - Average headline length: {df['headline_length'].mean():.1f} characters")
    print(f"   - Unique headlines: {df['headline'].nunique():,}")
    print()

if 'stock' in df.columns:
    print(f"3. Stock Coverage:")
    print(f"   - Unique stocks mentioned: {df['stock'].nunique():,}")
    print(f"   - Top 5 stocks: {', '.join(df['stock'].value_counts().head(5).index.tolist())}")
    print()

if date_col:
    print(f"4. Temporal Coverage:")
    print(f"   - Date range: {df[date_col].min().strftime('%Y-%m-%d')} to {df[date_col].max().strftime('%Y-%m-%d')}")
    print(f"   - Total days: {(df[date_col].max() - df[date_col].min()).days:,}")
    print(f"   - Average articles per day: {len(df) / ((df[date_col].max() - df[date_col].min()).days + 1):.1f}")
    print()

print(f"5. Data Quality:")
total_missing = df.isnull().sum().sum()
print(f"   - Total missing values: {total_missing:,}")
print(f"   - Duplicate rows: {df.duplicated().sum():,}")
print()

print("6. Recommendations for Next Steps:")
print("   - Clean missing values in critical columns")
print("   - Consider text preprocessing for sentiment analysis")
print("   - Prepare dataset for technical indicator computation")
print("   - Set up data pipeline for Task 2")

## 11. Save Cleaned Dataset

In [ ]:
# Save the cleaned dataset for next tasks
output_path = '../data/raw/cleaned_analyst_ratings.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"Final dataset shape: {df.shape}")

# Also save a smaller sample for quick testing
sample_size = min(10000, len(df))
sample_df = df.sample(n=sample_size, random_state=42)
sample_path = '../data/raw/sample_analyst_ratings.csv'
sample_df.to_csv(sample_path, index=False)
print(f"Sample dataset ({sample_size:,} rows) saved to: {sample_path}")

---

**Task 1 Complete!** 

Next steps:
1. Commit this notebook to the task-1 branch
2. Review findings and insights
3. Prepare for Task 2: Technical Indicators
4. Set up sentiment analysis pipeline